In [1]:
%pip install trl
import sys
!{sys.executable} -m pip install --upgrade --no-cache-dir "jinja2>=3.1.6"

from importlib.metadata import version
print(version("jinja2"))

import jinja2
print(jinja2.__version__)

# %conda install -y -c conda-forge compilers
!which gcc
!which clang

import os
os.environ["CC"] = "/opt/conda/bin/gcc"
! /opt/conda/bin/gcc --version




Note: you may need to restart the kernel to use updated packages.
3.1.6
3.1.6
/opt/conda/bin/gcc
gcc (conda-forge gcc 14.2.0-2) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [2]:
import torch
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

use_cuda = torch.cuda.is_available()
print(f"GPU available: {torch.cuda.is_available()}")

!nvidia-smi

GPU available: True
Thu Nov 19 01:33:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.108                Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1050        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   38C    P8            N/A  / 5001W |       0MiB /   3072MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [3]:
from huggingface_hub import notebook_login

notebook_login()


In [4]:

import torch
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
OUTPUT_MODEL = f"./{MODEL}_dpo"

dataset = load_dataset(
    "trl-lib/ultrafeedback_binarized",
    split="train",
)

args = DPOConfig(
    output_dir=OUTPUT_MODEL,
    bf16=False,
    fp16=use_cuda,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_length=128,
    logging_steps=10,
    
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    use_cpu=not use_cuda,
)

trainer = DPOTrainer(
    model=MODEL,
    train_dataset=dataset,
    args=args,
)

# trainer.train()
trainer.train(resume_from_checkpoint=True)
trainer.save_model(OUTPUT_MODEL)



/opt/conda/lib/python3.11/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss
68010,0.519969
68020,0.667215
68030,1.899301
68040,0.484311
68050,2.825689
68060,0.391864
68070,1.785116
68080,1.047135
68090,1.684955
68100,1.046001


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (356 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Voir seconde partie avec le notebook analyser_qualitativement.ipynb
